# 04 — Predicción sobre el dataset de prueba

**Proyecto 1: Competencia de modelación** · CC3092 Deep Learning y sistemas inteligentes

Notebook operativo para el día de la presentación. Consume un CSV con la estructura
de `pipeline_test.csv`, aplica el pipeline entrenado y genera el archivo de
predicciones en el formato de `expected_output.csv`.

**Para usarlo el lunes:** cambiar `ARCHIVO_PRUEBA` por la ruta del dataset nuevo y
ejecutar todas las celdas. No requiere reentrenar ni ningún paso manual.

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from config import EXPECTED_OUTPUT_CSV, ID_COL, PIPELINE_TEST_CSV, PRED_COL, TARGET
from predict import load_artifacts, predict_dataframe
from utils import load_csv, rmse

# >>> EL LUNES: cambiar esta ruta por la del dataset de prueba entregado <<<
ARCHIVO_PRUEBA = PIPELINE_TEST_CSV
ARCHIVO_SALIDA = Path.cwd().parent / "submission.csv"

## 1. Carga de los artefactos entrenados

Se cargan los pesos, la arquitectura y —crítico— los **parámetros exactos del
preprocesamiento** de cada miembro del ensemble: las medianas de imputación, las
categorías vistas y las medias/desviaciones de normalización.

Es el requisito 2 de las recomendaciones del proyecto: el preprocesamiento aplicado
al dataset de prueba debe ser idéntico al del entrenamiento. Al venir serializado,
no puede diferir.

In [2]:
bundle = load_artifacts()

print(f"Miembros del ensemble : {len(bundle['loaded_models'])}")
print(f"Features de entrada   : {bundle['n_features']}")
print(f"Rango de predicción   : {bundle['price_floor']:,.0f} – {bundle['price_ceiling']:,.0f} USD")
print(f"RMSE esperado (OOF)   : {bundle['oof_rmse']:,.0f} USD")

Miembros del ensemble : 15
Features de entrada   : 250
Rango de predicción   : 17,450 – 440,563 USD
RMSE esperado (OOF)   : 26,473 USD


## 2. Carga del dataset de prueba

`utils.load_csv` normaliza las diferencias de formato observadas entre `train.csv` y
el archivo de muestra del profesor:

- Valores categóricos con comillas simples (`'Wd Shng'`) que crearían una categoría
  distinta a la vista en entrenamiento.
- Columnas numéricas que vienen como entero en el test y float en train.
- Celdas vacías vs. el literal `NA`.

In [3]:
df_test = load_csv(ARCHIVO_PRUEBA)

print(f"Archivo: {ARCHIVO_PRUEBA.name}")
print(f"Filas:   {df_test.shape[0]}")
print(f"Columnas: {df_test.shape[1]}")
print(f"¿Trae SalePrice?: {TARGET in df_test.columns}")

df_test.head()

Archivo: pipeline_test.csv
Filas:   5
Columnas: 80
¿Trae SalePrice?: False


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,893,20,RL,70,8414,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,2,2006,WD,Normal
1,1106,60,RL,98,12256,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
2,414,30,RM,56,8960,Pave,Grvl,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,3,2010,WD,Normal
3,523,50,RM,50,5000,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,10,2006,WD,Normal
4,1037,20,RL,89,12898,Pave,NaN,IR1,HLS,AllPub,...,0,0,NaN,NaN,NaN,0,9,2009,WD,Normal


### Verificación de compatibilidad

Antes de predecir, comprobar que el archivo trae lo que el pipeline espera.

In [4]:
from config import TRAIN_CSV

df_train = load_csv(TRAIN_CSV)
esperadas = set(df_train.columns) - {TARGET}
recibidas = set(df_test.columns)

faltantes = esperadas - recibidas
extras = recibidas - esperadas

print(f"Columnas faltantes: {sorted(faltantes) if faltantes else 'ninguna'}")
print(f"Columnas extra:     {sorted(extras) if extras else 'ninguna'}")
print(f"Columna {ID_COL} presente: {ID_COL in df_test.columns}")

if not faltantes:
    print("\n-> El archivo es compatible con el pipeline entrenado.")
else:
    print("\n-> Las columnas faltantes se imputarán con el valor por defecto del")
    print("   preprocesador (mediana para numéricas, 'None' para categóricas).")

Columnas faltantes: ninguna
Columnas extra:     ninguna
Columna Id presente: True

-> El archivo es compatible con el pipeline entrenado.


## 3. Predicción

El pipeline completo por cada miembro del ensemble:

1. Imputación de nulos según su significado.
2. Codificación ordinal y one-hot con las **categorías fijadas en entrenamiento**.
3. Features derivadas (`TotalSF`, `HouseAge`, interacciones…).
4. `log1p` sobre las numéricas sesgadas y estandarización z-score.
5. Predicción del MLP en escala logarítmica y conversión a precio.
6. Corrección de smearing.

Luego se promedian los 15 miembros en escala logarítmica (media geométrica, que es
la escala en la que se optimizó la pérdida) y se acota al rango plausible.

In [5]:
predicciones = predict_dataframe(df_test)
ids = df_test[ID_COL].to_numpy()

resultado = pd.DataFrame({ID_COL: ids, PRED_COL: np.round(predicciones, 2)})
resultado

,Id,Prediction
0,893,148737.437500
1,1106,324601.125000
2,414,99439.640625
3,523,162914.437500
4,1037,320043.531250


## 4. Verificación del formato de salida

El formato debe ser idéntico al de `expected_output.csv`, y la columna `Id` debe ser
la misma del archivo de entrada.

In [6]:
esperado = pd.read_csv(EXPECTED_OUTPUT_CSV)

print(f"Columnas esperadas : {list(esperado.columns)}")
print(f"Columnas generadas : {list(resultado.columns)}")
print(f"Coinciden          : {list(esperado.columns) == list(resultado.columns)}")
print()
print(f"Filas esperadas    : {len(esperado)}")
print(f"Filas generadas    : {len(resultado)}")
print()

if len(esperado) == len(resultado):
    mismo_orden = (esperado[ID_COL].to_numpy() == resultado[ID_COL].to_numpy()).all()
    print(f"Ids en el mismo orden: {mismo_orden}")

print(f"\n¿Alguna predicción nula o inválida?: "
      f"{resultado[PRED_COL].isna().any() or (resultado[PRED_COL] <= 0).any()}")

Columnas esperadas : ['Id', 'Prediction']
Columnas generadas : ['Id', 'Prediction']
Coinciden          : True

Filas esperadas    : 5
Filas generadas    : 5

Ids en el mismo orden: True

¿Alguna predicción nula o inválida?: False


### Revisión de cordura de las predicciones

Comparar la distribución predicha contra la del entrenamiento: si el modelo
devolviera precios fuera de rango o concentrados en un punto, se vería aquí.

In [7]:
comparacion = pd.DataFrame({
    "entrenamiento": df_train[TARGET].describe(),
    "predicciones": resultado[PRED_COL].describe(),
}).round(0)
comparacion

,entrenamiento,predicciones
count,1168.0,5.0
mean,181442.0,211147.0
std,77264.0,104200.0
min,34900.0,99440.0
25%,130000.0,148737.0
50%,165000.0,162914.0
75%,214925.0,320044.0
max,745000.0,324601.0


## 5. Escritura del archivo

In [8]:
resultado.to_csv(ARCHIVO_SALIDA, index=False)
print(f"Archivo escrito: {ARCHIVO_SALIDA}")
print(f"({len(resultado)} predicciones)\n")

print(ARCHIVO_SALIDA.read_text())

Archivo escrito: /Users/ferahmz/Documents/uvg/octavo/Deep Learning/proy1-dl/submission.csv
(5 predicciones)

Id,Prediction
893,148737.44
1106,324601.12
414,99439.64
523,162914.44
1037,320043.53



## 6. Cálculo del RMSE

Si el archivo de entrada incluye la columna `SalePrice`, se calcula el RMSE
directamente. Con el archivo de muestra no aplica, porque no trae los precios reales.

In [9]:
if TARGET in df_test.columns:
    score = rmse(df_test[TARGET].to_numpy(dtype=float), predicciones)
    print(f"RMSE sobre el archivo de entrada: {score:,.2f} USD")
else:
    print("El archivo de prueba no incluye SalePrice, así que no se puede calcular")
    print("el RMSE localmente.\n")
    print(f"RMSE estimado a partir de la validación del modelo:")
    print(f"  validación cruzada (out-of-fold) : {bundle['oof_rmse']:,.0f} USD")
    print(f"  holdout independiente             : 28,091 USD")

El archivo de prueba no incluye SalePrice, así que no se puede calcular
el RMSE localmente.

RMSE estimado a partir de la validación del modelo:
  validación cruzada (out-of-fold) : 26,473 USD
  holdout independiente             : 28,091 USD


---

## Equivalente desde la terminal

El mismo procedimiento sin abrir el notebook:

```bash
python src/predict.py --input <dataset_de_prueba.csv> --output submission.csv
```